# Chapter 1 &mdash; Three Machines, One Idea: Restrictions of the Turing Machine

**Concept 14 of the Chapter 1 decomposition:** *FA, PDA and LBA as Simplified Turing Machines*

Finite automata, pushdown automata and linear bounded automata are not separate inventions &mdash; they are <b>restrictions</b> of the Turing machine, one per pattern class.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-FA-PDA-LBA-As-Simplified-TMs/Concept-FA-PDA-LBA-As-Simplified-TMs.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_PDA        import *
from jove.Def_TM         import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


| Machine | Memory | Recognises |
|---|---|---|
| **FA** (Rabin &amp; Scott, 1957) | none beyond the state | regular |
| **PDA** (Ginsburg, Greibach, 1960s) | one unbounded **stack** | context-free |
| **LBA** (Kuroda, 1960s) | tape **snipped** to the input | context-sensitive |
| **TM** (Turing, 1936) | unbounded tape | recursively enumerable |

Note the chronology: **the most general machine came first**, and the restrictions
arrived twenty years later, driven by practical parsing needs.

## 2. Definitions

### The same language, three ways: $a^n b^n$

A DFA cannot do it. A PDA can. A TM certainly can. Let us watch the DFA fail.

In [ ]:
# A DFA that tries to check a^n b^n -- and can only manage n <= 2
# Again mind the naming rule: the ACCEPTING state must start with 'F',
# and 'IF' means initial AND final (so the empty string is accepted).
tryab = md2mc('''DFA
IF : a -> A1
IF : b -> BH
A1 : a -> A2
A1 : b -> Fdone
A2 : a -> BH
A2 : b -> B1
B1 : b -> Fdone
B1 : a -> BH
Fdone : a | b -> BH
BH : a | b -> BH
''')
print("bounded a^n b^n DFA, states :", sorted(tryab["Q"]))
print("final states :", sorted(tryab["F"]), " <- must be non-empty, or it accepts nothing!")
assert tryab["F"], "a DFA with no F-named state accepts NOTHING -- a silent bug"

### The PDA: push each `a`, pop one per `b`

In [ ]:
anbn = md2mc('''PDA
IF : a , #  ; a#  -> Pa
Pa : a , a  ; aa  -> Pa
Pa : b , a  ; ''  -> Pb
Pb : b , a  ; ''  -> Pb
Pb : '' , # ; #   -> F
''')
print("a^n b^n PDA states :", sorted(anbn["Q"]))

## 3. Tests

The DFA works up to its built-in bound, then breaks.

In [ ]:
for n in range(0, 5):
    s = 'a'*n + 'b'*n
    print("n=%d  %-10s DFA says %s" % (n, s, accepts_dfa(tryab, s)))
assert accepts_dfa(tryab, "aabb") and not accepts_dfa(tryab, "aaabbb")
print()
print("n=3 fails -- the DFA ran out of states, not out of legality.")
assert accepts_dfa(tryab, "aabb") and not accepts_dfa(tryab, "aaabbb")

The PDA has no such bound: the stack grows with the input.

In [ ]:
explore_pda("aaabbb", anbn, STKMAX=8)

The restriction ladder, as programming restrictions on C.

In [ ]:
print("FA  : finitely many FINITE variables, no heap, no recursion")
print("PDA : + functions that may recurse   (the call stack IS the stack)")
print("LBA : + unbounded memory, but only as much tape as the input")
print("TM  : + unbounded memory, freely accessed")
print()
print("CAUTION: TWO stacks, or ONE queue, already give full TM power.")
print("The 'single stack' restriction is what keeps a PDA a PDA.")

## 4. Animation


The finite automaton: no memory beyond its current state. Watch it &mdash; there is
nowhere for a count to live.


*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(tryab, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Extend `tryab` to handle $n \le 4$. How many states per extra level?
2. The sidenote says two stacks give TM power. Sketch how two stacks simulate a tape.
   (Chapter 13 does this properly.)
3. Which machine would you need for "the same number of `a`s, `b`s **and** `c`s"?
   Try a PDA and see where the single stack runs out.

In [ ]:
# Your work for the exercises above.